# HDD 14 - Q3 multi-model failure-warning evaluation foundation

This notebook scores the five Notebook 13 development cohorts with two **predeclared SMART baselines**: (1) positive reallocated/pending/offline-uncorrectable sector count, and (2) positive one-day increases in those counters. It evaluates recorded Q3 failures using the saved first-event audit. The previously trained neural detector was **not fitted or scored on these fresh five-model drives**; its performance is not produced by this notebook. Q4 reservations and June stay closed.

Run in the same environment as Notebook 13. The Q3 ZIP is reused. This can take time and disk space: it streams all 92 daily files and writes scored August/September partitions. No model is trained and no threshold is selected from failures.

In [1]:
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import hashlib, json, re, zipfile
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.metrics import average_precision_score

COMPONENT_ROOT = Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID = 'single_model_q1q2_2021_v1'
AUDIT_ID = 'q3_multimodel_event_coverage_v1'
EVAL_ID = 'q3_multimodel_warning_baselines_v1'
Q3_ZIP = COMPONENT_ROOT / 'data/raw/data_Q3_2021.zip'
AUDIT = COMPONENT_ROOT / 'results' / RUN_ID / AUDIT_ID
OUT = COMPONENT_ROOT / 'results' / RUN_ID / EVAL_ID
PARTS = OUT / 'daily_scores'
OUT.mkdir(parents=True, exist_ok=True); PARTS.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def load_json(path):
    assert path.is_file(), f'Missing input: {path}'
    return json.loads(path.read_text())

completion_path = AUDIT / 'multimodel_coverage_complete.json'
completion = load_json(completion_path)
assert completion['analysis_complete'] and not completion['future_test_outcomes_opened'] and not completion['sealed_June_test_opened']
reservation_path = COMPONENT_ROOT / completion['split_path']
assert sha256(reservation_path) == completion['split_sha256']
reservation = load_json(reservation_path)
assert reservation['future_test_outcomes_opened'] is False
models = list(reservation['models'])
assert len(models) == 5 and set(models) == set(completion['provisional_models'])
dev_model = {str(s): model for model, groups in reservation['models'].items() for s in groups['development']}
reserved = {str(s) for groups in reservation['models'].values() for s in groups['future_test']}
assert not set(dev_model) & reserved
assert Q3_ZIP.is_file() and sha256(Q3_ZIP) == completion['settings']['archive_sha256']
event_path = COMPONENT_ROOT / 'data/evaluation_only' / RUN_ID / AUDIT_ID / 'development_first_failure_support.csv'
events = pd.read_csv(event_path, dtype={'serial_number':str}, parse_dates=['failure_date'])
assert not events.serial_number.duplicated().any()
assert set(events.serial_number) <= set(dev_model) and not set(events.serial_number) & reserved
assert all(dev_model[s] == m for s,m in events[['serial_number','model']].itertuples(index=False, name=None))
first_failure = {s:pd.Timestamp(d) for s,d in events[['serial_number','failure_date']].itertuples(index=False,name=None)}
DATES = pd.date_range('2021-07-01','2021-09-30')
DAY = {str(date.date()):i for i,date in enumerate(DATES)}
FEATURES = ['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
SECTOR = ['smart_5_raw','smart_197_raw','smart_198_raw']
SETTINGS = {'calibration_dates':['2021-08-01','2021-08-15'],
            'evaluation_dates':['2021-08-16','2021-09-30'],
            'events_evaluated':['2021-09-01','2021-09-30'],
            'horizons_days':[7,14,30],
            'score_definition':{'sector_positive':'any of SMART 5/197/198 positive at endpoint',
                                'sector_delta':'sum of positive one-day changes in SMART 5/197/198'},
            'alarm_rule':'score > 0; fixed before label evaluation',
            'window_days':30,
            'population':'Notebook 13 July-1 declared development drives only',
            'reservation_sha256':sha256(reservation_path),
            'event_support_sha256':sha256(event_path),
            'q3_zip_sha256':sha256(Q3_ZIP)}
protocol = OUT / 'protocol.json'
if protocol.exists(): assert load_json(protocol) == SETTINGS, 'Protocol changed. Use a new EVAL_ID.'
else: protocol.write_text(json.dumps(SETTINGS, indent=2))
print('Development drives:',len(dev_model),'| reserved Q4 drives excluded:',len(reserved),'| recorded Q3 events:',len(events))

Development drives: 77412 | reserved Q4 drives excluded: 19351 | recorded Q3 events: 230


## 1. Stream complete 30-day windows and calculate fixed SMART scores

A missing date or invalid five-feature reading resets a drive's 30-day streak. Failure flags are never read in this step. Only development drives are loaded. Score partitions include calendar dates from August 1; August 1–15 is a descriptive calibration period, and August 16–September 30 supplies evaluation endpoints. Baseline alarm rules are already fixed at `score > 0`.

In [2]:
with zipfile.ZipFile(Q3_ZIP) as z:
    members = {}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename: continue
        match = re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in DAY:
            assert match.group(1) not in members, 'Duplicate daily file'
            members[match.group(1)] = info.filename
assert set(members) == set(DAY), 'Missing Q3 daily file'

# State is bounded by the declared development-drive count, not the full ZIP size.
last_day = {}; streak = {}; prior_sector = {}; observed_bits = {}; valid_bits = {}
partition_hashes = {}; quality_counts = {m:0 for m in models}
for date, member in sorted(members.items()):
    d = DAY[date]; day_rows = []; seen = set()
    with zipfile.ZipFile(Q3_ZIP) as z, z.open(member) as f:
        for chunk in pd.read_csv(f, usecols=['date','serial_number']+FEATURES,
                                 dtype={'serial_number':str}, chunksize=50000):
            part = chunk.loc[chunk.serial_number.isin(dev_model)].copy()
            if part.empty: continue
            assert part.date.astype(str).eq(date).all(), 'Filename/date mismatch'
            serials = part.serial_number.tolist()
            assert len(serials)==len(set(serials)) and seen.isdisjoint(serials), 'Duplicate drive/date'
            seen.update(serials)
            values = part[FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64)
            for serial, v in zip(serials,values):
                observed_bits[serial] = observed_bits.get(serial,0) | (1 << d)
                valid = bool(np.isfinite(v).all() and (v>=0).all())
                if not valid:
                    streak[serial]=0;prior_sector.pop(serial,None);last_day[serial]=d
                    continue
                consecutive = last_day.get(serial,-2)==d-1
                streak[serial] = (streak.get(serial,0)+1 if consecutive else 1)
                old_sector = prior_sector.get(serial) if consecutive else None
                last_day[serial]=d
                sector = v[[0,3,4]]
                prior_sector[serial] = sector
                if streak[serial]<30: continue
                valid_bits[serial] = valid_bits.get(serial,0) | (1 << d)
                quality_counts[dev_model[serial]] += 1
                if date < '2021-08-01': continue
                day_rows.append((serial,dev_model[serial],date,
                                 float(np.max(sector)),
                                 float(np.maximum(sector-old_sector,0).sum()) if old_sector is not None else 0.0))
    if date >= '2021-08-01':
        frame = pd.DataFrame(day_rows,columns=['serial_number','model','end_date','sector_positive','sector_delta'])
        assert not frame.duplicated(['serial_number','end_date']).any()
        path = PARTS / f'{date}.parquet'
        if path.exists():
            assert pd.read_parquet(path).equals(frame), f'Existing partition differs: {path}'
        else: frame.to_parquet(path,index=False)
        partition_hashes[path.name]=sha256(path)
    if date.endswith(('-01','-15')): print('Streamed:',date)
assert len(partition_hashes)==61
print('Valid 30-day endpoints across Q3:',sum(quality_counts.values()))
print('Score partitions:',len(partition_hashes))

Streamed: 2021-07-01
Streamed: 2021-07-15
Streamed: 2021-08-01
Streamed: 2021-08-15
Streamed: 2021-09-01
Streamed: 2021-09-15
Valid 30-day endpoints across Q3: 4848446
Score partitions: 61


## 2. Check coverage against Notebook 13

The two notebooks must count exactly the same 30-day feature windows per model. No result is reported if counts differ. Recorded events are joined only after feature scores and their input hashes are fixed.

In [3]:
coverage = pd.read_csv(AUDIT/'model_event_coverage_summary.csv')
for row in coverage.itertuples(index=False):
    assert quality_counts[row.model] == int(row.quality_30day_windows), (row.model,quality_counts[row.model],row.quality_30day_windows)
assert int(coverage.first_Q3_failure_events.sum()) == len(events)
score_manifest = {'protocol_sha256':sha256(protocol),'partition_sha256':partition_hashes,
                  'quality_windows':quality_counts,'reservation_sha256':sha256(reservation_path),
                  'q3_zip_sha256':sha256(Q3_ZIP)}
manifest_path=OUT/'score_manifest.json'
if manifest_path.exists(): assert load_json(manifest_path)==score_manifest, 'Score manifest changed.'
else: manifest_path.write_text(json.dumps(score_manifest,indent=2))
print('Notebook 13 coverage reproduced exactly. Feature scores frozen before label evaluation.')

Notebook 13 coverage reproduced exactly. Feature scores frozen before label evaluation.


## 3. Evaluate endpoints and first failure events

Positive endpoint: first recorded failure occurs 1–h days later. Negative endpoint: all following h dates were observed and no failure occurred in that interval. Other endpoints are censored. Event recall requires **every** one of the h pre-failure dates to have a valid score, so early Q3 failures and missing data cannot silently become misses. Overlapping windows are not independent. These baseline alert rules can have a high alert burden; they are not matched-budget comparisons.

In [4]:
def has_all_bits(bits,lo,hi):
    if lo<0 or hi>=len(DATES): return False
    mask=((1<<(hi-lo+1))-1)<<lo
    return bits & mask == mask

def score_metrics(model, baseline, horizon, frame):
    history=frame.loc[frame.model.eq(model)].copy()
    subset=history.loc[history.end_date.ge('2021-08-16')].copy()
    if subset.empty: return None, pd.DataFrame()
    subset['end_date']=pd.to_datetime(subset.end_date)
    score=subset[baseline].to_numpy(dtype=float)
    flags=score>0
    serials=subset.serial_number.to_numpy(); endpoints=subset.end_date.to_numpy()
    targets=np.full(len(subset),np.nan)
    for i,(serial,end) in enumerate(zip(serials,endpoints)):
        day=DAY[str(pd.Timestamp(end).date())]
        fail=first_failure.get(serial)
        failure_day=DAY[str(fail.date())] if fail is not None else None
        if failure_day is not None and failure_day<=day: continue
        if failure_day is not None and failure_day-day<=horizon: targets[i]=1
        elif has_all_bits(observed_bits.get(serial,0),day+1,day+horizon): targets[i]=0
    keep=np.isfinite(targets); y=targets[keep].astype(int); s=score[keep]; f=flags[keep]
    positive=int((y==1).sum()); negative=int((y==0).sum())
    tp=int((f & (y==1)).sum()); fp=int((f & (y==0)).sum())
    history['end_date']=pd.to_datetime(history.end_date)
    history['flag']=history[baseline].to_numpy(dtype=float)>0
    by_drive={serial:g.set_index('end_date') for serial,g in history.groupby('serial_number',sort=False)}
    details=[]
    for event in events.loc[events.model.eq(model)].itertuples(index=False):
        fail=pd.Timestamp(event.failure_date)
        if not pd.Timestamp('2021-09-01')<=fail<=pd.Timestamp('2021-09-30'): continue
        required=pd.date_range(end=fail-pd.Timedelta(days=1),periods=horizon)
        g=by_drive.get(event.serial_number)
        complete=bool(g is not None and all(x in g.index for x in required))
        alarms=g.loc[required,'flag'] if complete else pd.Series(dtype=bool)
        first_alarm=required[np.flatnonzero(alarms.to_numpy())[0]] if complete and alarms.any() else pd.NaT
        details.append({'serial_number':event.serial_number,'model':model,'baseline':baseline,
                        'horizon_days':horizon,'failure_date':fail,'complete_warning_history':complete,
                        'recalled':bool(complete and alarms.any()),
                        'first_warning_lead_days':int((fail-first_alarm).days) if pd.notna(first_alarm) else np.nan})
    ed=pd.DataFrame(details)
    complete=ed.loc[ed.complete_warning_history] if len(ed) else ed
    recalled=complete.loc[complete.recalled] if len(complete) else complete
    row={'model':model,'baseline':baseline,'horizon_days':horizon,'scored_endpoints':len(subset),
         'evaluable_endpoints':len(y),'positive_endpoints':positive,'negative_endpoints':negative,
         'censored_or_post_failure':int((~keep).sum()),
         'average_precision':float(average_precision_score(y,s)) if positive and negative else np.nan,
         'positive_prevalence':float(y.mean()) if len(y) else np.nan,
         'threshold_precision':tp/(tp+fp) if tp+fp else np.nan,
         'positive_endpoint_recall':tp/positive if positive else np.nan,
         'false_alerts_per_1000_observed_negative_endpoints':1000*fp/negative if negative else np.nan,
         'all_scored_alerts_per_1000_endpoints':1000*flags.mean(),
         'September_recorded_events':len(ed),'events_with_complete_warning_history':len(complete),
         'events_excluded_partial_history':len(ed)-len(complete),
         'events_recalled':len(recalled),
         'strict_event_recall':len(recalled)/len(complete) if len(complete) else np.nan,
         'median_first_warning_lead_days':float(recalled.first_warning_lead_days.median()) if len(recalled) else np.nan,
         'status':'both_endpoint_classes' if positive and negative else 'insufficient_endpoint_classes'}
    return row,ed

# August 1–15 supplies warning history for early September events; endpoint
# metrics still begin August 16 and do not fit thresholds on those dates.
frames=[pd.read_parquet(PARTS/f'{date}.parquet') for date in DAY if date>='2021-08-01']
scored=pd.concat(frames,ignore_index=True)
assert not scored.duplicated(['serial_number','end_date']).any()
assert set(scored.serial_number)<=set(dev_model) and not set(scored.serial_number)&reserved
rows=[]; event_details=[]
for model in models:
    for baseline in ['sector_positive','sector_delta']:
        for horizon in SETTINGS['horizons_days']:
            row,details=score_metrics(model,baseline,horizon,scored)
            if row:rows.append(row);event_details.append(details)
metrics=pd.DataFrame(rows)
assert len(metrics)==len(models)*2*3
metrics.to_csv(OUT/'q3_baseline_failure_metrics.csv',index=False)
pd.concat(event_details,ignore_index=True).to_csv(OUT/'q3_baseline_event_details.csv',index=False)
display(metrics[['model','baseline','horizon_days','positive_endpoints','average_precision',
                 'events_with_complete_warning_history','strict_event_recall',
                 'false_alerts_per_1000_observed_negative_endpoints']])
print('These are fixed SMART baseline results, not neural detector results.')

C:\Users\IMAKA\AppData\Local\Temp\ipykernel_22904\4034382647.py:32: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  required=pd.date_range(end=fail-pd.Timedelta(days=1),periods=horizon)
C:\Users\IMAKA\AppData\Local\Temp\ipykernel_22904\4034382647.py:32: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  required=pd.date_range(end=fail-pd.Timedelta(days=1),periods=horizon)
C:\Users\IMAKA\AppData\Local\Temp\ipykernel_22904\4034382647.py:32: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  required

,model,baseline,horizon_days,positive_endpoints,average_precision,events_with_complete_warning_history,strict_event_recall,false_alerts_per_1000_observed_negative_endpoints
0,TOSHIBA MG07ACA14TA,sector_positive,7,161,0.002174,16,0.2500,7.770813
1,TOSHIBA MG07ACA14TA,sector_positive,14,293,0.004877,16,0.2500,7.709262
2,TOSHIBA MG07ACA14TA,sector_positive,30,510,0.015074,16,0.2500,7.504356
3,TOSHIBA MG07ACA14TA,sector_delta,7,161,0.000519,16,0.1250,0.067163
4,TOSHIBA MG07ACA14TA,sector_delta,14,293,0.000587,16,0.1250,0.069640
5,TOSHIBA MG07ACA14TA,sector_delta,30,510,0.001520,16,0.1250,0.063534
6,ST12000NM0008,sector_positive,7,135,0.003425,16,0.3750,14.851858
7,ST12000NM0008,sector_positive,14,264,0.007856,16,0.3750,14.643606
8,ST12000NM0008,sector_positive,30,528,0.018088,16,0.3750,14.234221
9,ST12000NM0008,sector_delta,7,135,0.012326,16,0.2500,0.492554


These are fixed SMART baseline results, not neural detector results.


## 4. Audit boundaries and record next required inputs

Q3 data ends September 30. Late endpoints cannot have a complete 30-day negative follow-up within this ZIP. No Q4 data is accessed to fill that gap. Report `NaN` where a class or history is unavailable. These baseline results do not justify changing the frozen detector or selecting a threshold after seeing failure labels.

In [5]:
assert not has_all_bits((1<<len(DATES))-1,90,93)
assert has_all_bits(sum(1<<d for d in range(60,75)),60,74)
assert not has_all_bits(sum(1<<d for d in range(60,75) if d!=68),60,74)
assert not metrics.empty and metrics.horizon_days.isin([7,14,30]).all()
assert sha256(reservation_path)==SETTINGS['reservation_sha256']
assert sha256(Q3_ZIP)==SETTINGS['q3_zip_sha256']
assert {p.name:sha256(p) for p in PARTS.glob('*.parquet')}==partition_hashes
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'protocol_sha256':sha256(protocol),'score_manifest_sha256':sha256(manifest_path),
        'baseline_metrics_sha256':sha256(OUT/'q3_baseline_failure_metrics.csv'),
        'development_drives':len(dev_model),'reserved_future_test_drives':len(reserved),
        'recorded_q3_events':len(events),'models_provisional':models,
        'trained_detector_scored_on_new_cohort':False,
        'all_device_types_verified':bool(completion['all_device_types_verified']),
        'future_test_outcomes_opened':False,'sealed_June_test_opened':False,
        'next_steps':['Verify four provisional HDD model types and SMART semantics',
                      'Train/score detector on development data for these five models without failure labels',
                      'Freeze detector scores and compare with these baselines at equal alert budgets',
                      'Reserve Q4 for blind evaluation; do not use it to fill Q3 follow-up']}
(OUT/'baseline_evaluation_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 14 COMPLETE — fixed SMART baseline warning evaluation saved.')
print('Five-model trained-detector evaluation: pending model-specific score generation.')
print('Q4 outcomes unopened. Results:',OUT)

NOTEBOOK 14 COMPLETE — fixed SMART baseline warning evaluation saved.
Five-model trained-detector evaluation: pending model-specific score generation.
Q4 outcomes unopened. Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_multimodel_warning_baselines_v1
